# ⚛️ Shor's Algorithm — Implementasi Lengkap dengan Qiskit 2.3.0

---

**Tujuan:** Memfaktorisasi bilangan komposit $N = p \cdot q$ menggunakan algoritma kuantum Shor.

**Platform:** Jupyter Notebook · Qiskit 2.3.0 · Python 3.10+ · HPC Node

---

## 📌 Alur Keseluruhan

```
INPUT N
  │
  ▼
[KLASIK]  Pilih a coprime dengan N
  │
  ▼
[QUANTUM] Inisialisasi Register → Superposisi (Hadamard)
  │
  ▼
[QUANTUM] Oracle: Modular Exponentiation  f(x) = aˣ mod N
  │
  ▼
[QUANTUM] Quantum Fourier Transform (QFT)
  │
  ▼
[QUANTUM] Pengukuran → Estimasi Period r (Continued Fraction)
  │
  ▼
[KLASIK]  Validasi r → Ekstraksi p dan q via GCD
  │
  ▼
OUTPUT: N = p × q
```

> **Contoh:** $N = 15 = 3 \times 5$, dengan $a = 7$, period $r = 4$

---
## 📦 STEP 0 — Import Library

Import semua library yang dibutuhkan untuk keseluruhan program:
- **`qiskit`**: Framework utama quantum computing (versi 2.3.0)
- **`qiskit_aer`**: Simulator quantum berbasis noise/ideal (pengganti `qiskit.providers.aer`)
- **`numpy`**: Operasi numerik dan array
- **`matplotlib`**: Visualisasi circuit dan histogram
- **`math`**: Fungsi GCD, sqrt, dan operasi matematika lainnya
- **`fractions`**: Continued Fraction Expansion untuk estimasi period
- **`random`**: Pemilihan acak nilai `a`

In [3]:
# STEP 0: IMPORT LIBRARY
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.circuit.library import QFT, UnitaryGate
from qiskit.visualization import plot_histogram
from qiskit_aer import AerSimulator

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from math import gcd, isqrt, log2, ceil
from fractions import Fraction
import random, warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#0a0f1e', 'axes.facecolor': '#0d1526',
    'axes.edgecolor': '#1e3a5f', 'axes.labelcolor': '#94a3b8',
    'text.color': '#e2e8f0', 'xtick.color': '#64748b',
    'ytick.color': '#64748b', 'grid.color': '#1e2d40',
    'grid.alpha': 0.5, 'font.family': 'monospace',
})

import qiskit, qiskit_aer
print(f"Qiskit {qiskit.__version__}  |  Aer {qiskit_aer.__version__}  |  NumPy {np.__version__}")
print("Import selesai ✅")


Qiskit 2.3.0  |  Aer 0.17.2  |  NumPy 2.2.6
Import selesai ✅


In [4]:
%matplotlib inline

---
## 🔢 STEP 1 — Input & Validasi N

### Teori
Algoritma Shor dirancang untuk memfaktorisasi bilangan komposit $N = p \cdot q$, di mana $p$ dan $q$ adalah bilangan prima.

Sebelum masuk ke komputasi quantum, lakukan **validasi klasik** terhadap $N$:

| Kondisi | Aksi |
|---------|------|
| $N$ genap | Faktor langsung: $2$ dan $N/2$ |
| $N$ prima | Tidak perlu difaktorisasi |
| $N = a^b$ | Faktor dengan akar |
| $N$ komposit & ganjil | ✅ Lanjut ke Shor |

Jumlah qubit yang dibutuhkan:
$$n_{count} = 2 \cdot \lceil \log_2 N \rceil \quad \text{(register pengukuran)}$$
$$n_{target} = \lceil \log_2 N \rceil \quad \text{(register target)}$$

In [5]:
# STEP 1: INPUT & VALIDASI N
def is_prime(n):
    if n < 2: return False
    if n == 2: return True
    if n % 2 == 0: return False
    for i in range(3, isqrt(n) + 1, 2):
        if n % i == 0: return False
    return True

def is_perfect_power(n):
    for b in range(2, int(log2(n)) + 1):
        a = round(n ** (1/b))
        for c in [a-1, a, a+1]:
            if c > 1 and c ** b == n:
                return True, c, b
    return False, None, None

def validate_N(N):
    if N % 2 == 0:
        print(f"Faktor klasik (genap): 2 × {N//2}")
        return False, 2, N // 2
    if is_prime(N):
        print(f"N={N} adalah prima, tidak perlu difaktorisasi.")
        return False, None, None
    is_pp, a_pp, _ = is_perfect_power(N)
    if is_pp:
        print(f"N adalah perpangkatan bulat, faktor klasik: {a_pp}")
        return False, a_pp, a_pp

    n_count  = 2 * ceil(log2(N))
    n_target = ceil(log2(N))
    print(f"N={N} valid  |  n_count={n_count}  n_target={n_target}  total={n_count+n_target} qubit ✅")
    return True, n_count, n_target

N = 15   # <-- ganti nilai N di sini
valid, n_count, n_target = validate_N(N)


N=15 valid  |  n_count=8  n_target=4  total=12 qubit ✅


---
## 🎲 STEP 2 — Pilih Nilai `a` (Coprime dengan N)

### Teori
Pilih bilangan acak $a$ dengan syarat:
$$1 < a < N \quad \text{dan} \quad \gcd(a, N) = 1$$

**Kasus Beruntung:** Jika $\gcd(a, N) \neq 1$, maka kita langsung mendapatkan faktor dari $N$ **tanpa perlu quantum computing!**

**Kasus Normal:** Jika $\gcd(a, N) = 1$, maka $a$ dan $N$ adalah **coprime** — lanjutkan ke rangkaian quantum untuk mencari period $r$ dari fungsi:
$$f(x) = a^x \mod N$$

Fungsi ini bersifat **periodik** dengan period $r$ sehingga $a^r \equiv 1 \pmod{N}$.

In [6]:
# STEP 2: PILIH NILAI a (COPRIME DENGAN N)
def choose_a(N, a_fixed=None):
    a = a_fixed if a_fixed is not None else random.randint(2, N - 1)
    g = gcd(a, N)
    if g != 1:
        print(f"Lucky case! gcd({a},{N})={g}  →  {N} = {g} × {N//g}")
        return None
    print(f"a={a}  |  gcd({a},{N})=1  →  coprime ✅")
    print(f"{'x':>4} | {'f(x)':>5}   (f(x) = {a}^x mod {N})")
    for x in range(8):
        fx = pow(a, x, N)
        mark = " ← period r" if (x > 0 and fx == 1) else ""
        print(f"{x:>4} | {fx:>5}{mark}")
    return a

a = choose_a(N, a_fixed=7)


a=7  |  gcd(7,15)=1  →  coprime ✅
   x |  f(x)   (f(x) = 7^x mod 15)
   0 |     1
   1 |     7
   2 |     4
   3 |    13
   4 |     1 ← period r
   5 |     7
   6 |     4
   7 |    13


---
## ⚛️ STEP 3 — Bangun Oracle: Modular Exponentiation Gate

### Teori
Oracle quantum $U_a$ mengimplementasikan fungsi periodik:
$$U_a |x\rangle|0\rangle = |x\rangle|a^x \bmod N\rangle$$

Untuk implementasi pada simulator (kasus kecil $N$), kita bangun **unitary matrix** dari fungsi klasik $f(x) = a^x \bmod N$, kemudian bungkus menjadi gate quantum yang reversible.

**Struktur gate:**
- Input: $n_{count}$ qubit (register $x$) + $n_{target}$ qubit (register target)
- Output: $|x\rangle|a^x \bmod N\rangle$ — entangled state antara $x$ dan $f(x)$

> ⚠️ **Catatan:** Untuk $N$ besar pada hardware nyata, oracle ini diimplementasikan dengan controlled-modular-multiplication yang jauh lebih kompleks ($O(n^3)$ CNOT gates). Di sini kita gunakan pendekatan unitary langsung untuk simulator.

In [7]:
# STEP 3: ORACLE — MODULAR EXPONENTIATION GATE
def build_mod_exp_oracle(a, N, n_count, n_target):
    dim_x      = 2 ** n_count
    dim_target = 2 ** n_target
    dim_total  = dim_x * dim_target

    U = np.zeros((dim_total, dim_total), dtype=complex)
    for x in range(dim_x):
        fx = pow(a, x, N)
        for y in range(dim_target):
            y_out   = y ^ fx if fx < dim_target else y
            idx_in  = x * dim_target + y
            idx_out = x * dim_target + y_out
            U[idx_out, idx_in] = 1

    is_unitary = np.allclose(U @ U.conj().T, np.eye(dim_total))
    print(f"Oracle f(x)={a}^x mod {N}  |  dim={dim_total}×{dim_total}  |  unitary={is_unitary} ✅")
    return U

U_matrix = build_mod_exp_oracle(a, N, n_count, n_target)


Oracle f(x)=7^x mod 15  |  dim=4096×4096  |  unitary=True ✅


---
## 🌀 STEP 4 — Bangun Circuit Kuantum Shor Lengkap

### Teori
Circuit quantum Shor terdiri dari tiga bagian utama:

**1. Inisialisasi & Superposisi (Hadamard)**
$$|\psi_0\rangle = H^{\otimes n} |0\rangle^n |1\rangle = \frac{1}{\sqrt{2^n}} \sum_{x=0}^{2^n-1} |x\rangle |1\rangle$$

**2. Oracle — Modular Exponentiation**
$$|\psi_1\rangle = \frac{1}{\sqrt{2^n}} \sum_{x=0}^{2^n-1} |x\rangle |a^x \bmod N\rangle$$

**3. Quantum Fourier Transform (QFT)**
$$\text{QFT}|j\rangle = \frac{1}{\sqrt{2^n}} \sum_{k=0}^{2^n-1} e^{2\pi i jk/2^n} |k\rangle$$

Setelah QFT, amplitudo akan **terpusat** (constructive interference) pada nilai-nilai:
$$k = \frac{j \cdot 2^n}{r}, \quad j = 0, 1, 2, \ldots, r-1$$

sehingga pengukuran akan menghasilkan nilai $k$ yang mengandung informasi period $r$.

In [8]:
# STEP 4: BANGUN CIRCUIT QUANTUM SHOR LENGKAP
def build_shors_circuit(a, N, n_count, n_target, U_matrix):
    qr_count  = QuantumRegister(n_count,  name='counting')
    qr_target = QuantumRegister(n_target, name='target')
    cr        = ClassicalRegister(n_count, name='measure')
    qc        = QuantumCircuit(qr_count, qr_target, cr)

    # Init
    qc.x(qr_target[0])
    qc.barrier(label='Init')
    # Hadamard
    for q in qr_count: qc.h(q)
    qc.barrier(label='H⊗n')
    # Oracle
    qc.append(UnitaryGate(U_matrix, label=f'{a}^x mod {N}'),
               list(qr_count) + list(qr_target))
    qc.barrier(label='Oracle')
    # Inverse QFT
    qc.append(QFT(n_count, inverse=True, do_swaps=True, name='QFT†'), qr_count)
    qc.barrier(label='QFT†')
    # Ukur
    qc.measure(qr_count, cr)
    print(f"Circuit siap  |  qubit={qc.num_qubits}  depth={qc.depth()} ✅")
    return qc

qc_shor = build_shors_circuit(a, N, n_count, n_target, U_matrix)


Circuit siap  |  qubit=12  depth=5 ✅


In [ ]:
print(type(qc_shor))
qc_shor.draw("mpl")

<class 'qiskit.circuit.quantumcircuit.QuantumCircuit'>


---
## 🖼️ STEP 5 — Visualisasi Circuit Quantum

### Keterangan
Visualisasi circuit menampilkan tiga bagian utama:
- **Register `counting`** ($n_{count}$ qubit): Dimulai dari $|0\rangle$, dikenai Hadamard, lalu diukur
- **Register `target`** ($n_{target}$ qubit): Dimulai dari $|0\rangle$, qubit pertama di-flip ke $|1\rangle$
- **Barrier** (garis vertikal): Pemisah antar fase komputasi
- **Oracle $U_a$**: Gate unitary untuk $f(x) = a^x \bmod N$
- **QFT†**: Inverse Quantum Fourier Transform

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

fig = qc_shor.draw(output="mpl", fold=40)

fig.suptitle(
    f"Shor Circuit N={N} a={a} depth={qc_shor.depth()}",
    fontsize=12
)

plt.show()

: 

---
## 🚀 STEP 6 — Jalankan Simulasi Quantum (Aer Simulator)

### Teori
Kita menjalankan circuit pada **Aer Simulator** — simulator statevector ideal tanpa noise.

Setelah pengukuran, kita mendapatkan distribusi probabilitas atas semua kemungkinan hasil $k$. Hasil dengan probabilitas tinggi sesuai dengan:
$$k \approx \frac{j \cdot 2^{n_{count}}}{r}, \quad j = 0, 1, \ldots, r-1$$

**Proses transpilasi:** Circuit di-transpile ke gate set native simulator sebelum dijalankan, mengoptimalkan kedalaman circuit.

Parameter simulasi:
- `shots` = jumlah pengulangan pengukuran (semakin banyak → statistik lebih akurat)
- Backend: `AerSimulator` dengan method `statevector`

In [ ]:
# STEP 6: SIMULASI QUANTUM
SHOTS = 2048

simulator  = AerSimulator(method='statevector')
qc_t       = transpile(qc_shor, simulator, optimization_level=1)
counts     = dict(sorted(
    simulator.run(qc_t, shots=SHOTS).result().get_counts().items(),
    key=lambda x: x[1], reverse=True
))

print(f"Shots={SHOTS}  |  state unik={len(counts)}")
print(f"{'bitstring':>12} | {'k':>5} | {'prob':>6}")
for bs, c in list(counts.items())[:8]:
    print(f"{bs:>12} | {int(bs,2):>5} | {c/SHOTS:>6.3f}")


---
## 📡 STEP 7 — Estimasi Period `r` via Continued Fraction Expansion

### Teori
Dari pengukuran, kita mendapatkan nilai $k$ (dalam desimal). Nilai ini berkaitan dengan period $r$ melalui:
$$\frac{k}{2^{n_{count}}} \approx \frac{j}{r}$$

**Continued Fraction Expansion (CFE)** digunakan untuk menemukan $r$ dari aproksimasi rasional:
$$\phi = \frac{k}{2^n} = a_0 + \cfrac{1}{a_1 + \cfrac{1}{a_2 + \cdots}}$$

Denominator dari konvergen CFE yang memenuhi $a^r \equiv 1 \pmod{N}$ adalah kandidat period $r$.

**Kelipatan period:** Jika konvergen memberikan $r' = j \cdot r$, kita perlu membagi dengan $\gcd(r', \text{kandidat lain})$.

In [ ]:
# STEP 7: ESTIMASI PERIOD r (CONTINUED FRACTION EXPANSION)
def find_period(counts, n_count, N, a):
    dim = 2 ** n_count
    print(f"{'k':>6} | {'phi':>7} | {'r_cand':>7} | {'a^r mod N':>9} | ok")
    r_found = None
    for bs in list(counts.keys())[:12]:
        k = int(bs, 2)
        if k == 0: continue
        phi = k / dim
        rc  = Fraction(phi).limit_denominator(N).denominator
        chk = pow(a, rc, N)
        ok  = "✅" if chk == 1 else "—"
        print(f"{k:>6} | {phi:>7.4f} | {rc:>7} | {chk:>9} | {ok}")
        if chk == 1 and r_found is None:
            r_found = rc
        if r_found is None:
            for m in range(2, 8):
                if pow(a, rc * m, N) == 1:
                    r_found = rc * m
                    break
    print(f"\nr = {r_found}" + (f"  →  {a}^{r_found} mod {N} = {pow(a,r_found,N)} ✅" if r_found else "  ❌ tidak ditemukan"))
    return r_found

r = find_period(counts, n_count, N, a)


---
## ✅ STEP 8 — Validasi Period `r`

### Teori
Period $r$ harus memenuhi **tiga syarat** agar dapat digunakan untuk faktorisasi:

| Syarat | Kondisi | Jika Gagal |
|--------|---------|------------|
| **Period valid** | $a^r \equiv 1 \pmod{N}$ | Ulangi quantum |
| **r genap** | $r \bmod 2 = 0$ | Ulangi dengan $a$ baru |
| **Non-trivial** | $a^{r/2} \not\equiv -1 \pmod{N}$ | Ulangi dengan $a$ baru |

Jika $r$ memenuhi semua syarat, maka:
$$\gcd(a^{r/2} \pm 1, N) \neq 1 \quad \text{dan} \quad \neq N$$

yang berarti faktor non-trivial dari $N$ dapat ditemukan.

In [ ]:
# STEP 8: VALIDASI PERIOD r
def validate_period(r, a, N):
    c1 = pow(a, r, N) == 1
    c2 = r % 2 == 0
    x  = pow(a, r // 2, N) if c2 else None
    c3 = (x != N - 1) if c2 else False
    print(f"a^r mod N = 1  : {c1}")
    print(f"r genap        : {c2}")
    print(f"a^(r/2) ≠ N-1  : {c3}" + (f"  (x={x})" if x else ""))
    ok = c1 and c2 and c3
    print(f"\nr={r} {'valid ✅' if ok else 'tidak valid ❌'}")
    return ok, x

period_valid, x_val = validate_period(r, a, N) if r else (False, None)


---
## 🏆 STEP 9 — Ekstraksi Faktor p dan q via GCD

### Teori
Ini adalah langkah akhir klasik. Dengan $x = a^{r/2} \bmod N$, kita manfaatkan identitas aljabar:

$$(a^{r/2} - 1)(a^{r/2} + 1) = a^r - 1 \equiv 0 \pmod{N}$$

Artinya $N$ **membagi** $(x-1)(x+1)$, tetapi **tidak membagi** $x-1$ maupun $x+1$ secara tunggal (karena syarat non-trivial). Akibatnya:

$$p = \gcd(x - 1, N) \neq 1 \quad \text{dan} \quad q = \gcd(x + 1, N) \neq 1$$

$$\boxed{N = p \times q}$$

In [ ]:
# STEP 9: EKSTRAKSI FAKTOR p DAN q via GCD
def extract_factors(x_val, a, r, N):
    p = gcd(x_val - 1, N)
    q = gcd(x_val + 1, N)
    ok = p not in (1, N) and q not in (1, N) and p * q == N
    print(f"x = {a}^{r//2} mod {N} = {x_val}")
    print(f"p = gcd({x_val-1}, {N}) = {p}")
    print(f"q = gcd({x_val+1}, {N}) = {q}")
    print(f"\n{'🎉 ' if ok else '❌ '}{N} = {p} × {q}" + (" ✅" if ok else "  (trivial, ulangi dengan a lain)"))
    return p, q

if period_valid and x_val is not None:
    p_factor, q_factor = extract_factors(x_val, a, r, N)
else:
    p_factor = q_factor = None
    print("Period tidak valid, ulangi dengan a berbeda.")


---
## 📊 STEP 10 — Visualisasi Histogram Hasil Quantum

### Keterangan
Histogram menampilkan distribusi probabilitas hasil pengukuran register quantum. Sumbu-x adalah **nilai $k$** yang terukur (dalam basis desimal), sumbu-y adalah **probabilitas**.

Puncak histogram terjadi pada nilai:
$$k \approx \frac{j \cdot 2^{n_{count}}}{r}, \quad j = 0, 1, 2, \ldots, r-1$$

Untuk $r=4$ dan $2^{n_{count}} = 2^8 = 256$, puncak diharapkan di:
$$k = 0, \; 64, \; 128, \; 192$$

In [ ]:
# STEP 10: VISUALISASI HISTOGRAM
def plot_quantum_results(counts, n_count, N, a, r, p_factor, q_factor):
    dim    = 2 ** n_count
    SHOTS  = sum(counts.values())
    k_prob = {int(b, 2): c / SHOTS for b, c in counts.items()}
    peaks  = [round(j * dim / r) for j in range(r)] if r else []

    fig = plt.figure(figsize=(15, 9))
    fig.patch.set_facecolor('#050d1a')
    gs  = GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.35,
                   left=0.07, right=0.96, top=0.90, bottom=0.08)

    # ── Plot 1: Histogram distribusi ─────────────────────────────
    ax1 = fig.add_subplot(gs[0, :])
    ax1.set_facecolor('#0d1526')
    all_k = sorted(k_prob)
    colors = ['#22d3ee' if any(abs(k - pk) <= 2 for pk in peaks) else '#1d4ed8'
              for k in all_k]
    ax1.bar(all_k, [k_prob[k] for k in all_k],
            color=colors, width=max(1, dim//80), edgecolor='none', alpha=0.9)
    for pk in peaks:
        ax1.axvline(pk, color='#f59e0b', alpha=0.55, linewidth=1.2, linestyle='--')
        ax1.text(pk, k_prob.get(pk, 0) + 0.008, f'k={pk}',
                 ha='center', fontsize=8, color='#f59e0b', fontweight='bold')
    ax1.set_xlabel('k  (hasil pengukuran)', fontsize=10)
    ax1.set_ylabel('Probabilitas', fontsize=10)
    ax1.set_title(f"Distribusi Pengukuran  –  N={N}  a={a}  shots={SHOTS:,}  2ⁿ={dim}",
                  fontsize=11, fontweight='bold', color='#e2e8f0')
    ax1.set_xlim(-dim*0.02, dim*1.02)
    ax1.grid(axis='y', alpha=0.25)
    leg = [mpatches.Patch(color='#22d3ee', label='Puncak teoritik'),
           mpatches.Patch(color='#1d4ed8', label='Lainnya')]
    ax1.legend(handles=leg, facecolor='#0d1526', edgecolor='#334155',
               labelcolor='#e2e8f0', fontsize=9)

    # ── Plot 2: Kandidat period dari CFE ─────────────────────────
    ax2 = fig.add_subplot(gs[1, 0])
    ax2.set_facecolor('#0d1526')
    top_k = sorted(k_prob, key=k_prob.get, reverse=True)[:8]
    r_cands, labels = [], []
    for k in top_k:
        if k == 0:
            r_cands.append(0); labels.append('k=0'); continue
        rc  = Fraction(k / dim).limit_denominator(N).denominator
        chk = pow(a, rc, N)
        r_cands.append(rc)
        labels.append(f'k={k}  r={rc} {"✓" if chk==1 else "✗"}')
    ax2.barh(range(len(top_k)), r_cands,
             color=['#22d3ee' if rc == r else '#334155' for rc in r_cands],
             edgecolor='none', alpha=0.85)
    ax2.set_yticks(range(len(top_k)))
    ax2.set_yticklabels(labels, fontsize=8)
    if r: ax2.axvline(r, color='#f59e0b', linewidth=1.8, linestyle='--', label=f'r={r}')
    ax2.set_xlabel('Kandidat r', fontsize=10)
    ax2.set_title('CFE → Kandidat Period', fontsize=10, fontweight='bold', color='#e2e8f0')
    ax2.legend(facecolor='#0d1526', edgecolor='#334155', labelcolor='#f59e0b', fontsize=9)
    ax2.grid(axis='x', alpha=0.25)

    # ── Plot 3: Ringkasan ─────────────────────────────────────────
    ax3 = fig.add_subplot(gs[1, 1])
    ax3.set_facecolor('#0d1526'); ax3.axis('off')
    rows = [
        ('N',           str(N),                         '#94a3b8'),
        ('a',           str(a),                         '#94a3b8'),
        ('Qubit',       f'{n_count+n_target}  ({n_count}+{n_target})', '#94a3b8'),
        ('r',           str(r),                         '#22d3ee'),
        ('x=a^(r/2)',   str(x_val),                     '#22d3ee'),
        ('p',           str(p_factor),                  '#84cc16'),
        ('q',           str(q_factor),                  '#84cc16'),
        ('N = p×q',     f'{p_factor} × {q_factor} = {p_factor*q_factor if p_factor else "?"}', '#f59e0b'),
    ]
    ax3.text(0.5, 1.0, 'Ringkasan', ha='center', va='top', fontsize=11,
             color='#e2e8f0', fontweight='bold', transform=ax3.transAxes)
    y = 0.88
    for label, val, clr in rows:
        ax3.text(0.08, y, f'{label}:', ha='left', fontsize=9.5,
                 color='#64748b', transform=ax3.transAxes)
        ax3.text(0.42, y, val, ha='left', fontsize=9.5,
                 color=clr, fontweight='bold', transform=ax3.transAxes)
        y -= 0.10
    box = mpatches.FancyBboxPatch((0.05, 0.03), 0.90, 0.10,
                                   boxstyle='round,pad=0.01', linewidth=1.5,
                                   edgecolor='#84cc16', facecolor='#1a2e05',
                                   transform=ax3.transAxes, zorder=3)
    ax3.add_patch(box)
    ax3.text(0.5, 0.08, f'🎉  {N} = {p_factor} × {q_factor}',
             ha='center', fontsize=13, color='#84cc16', fontweight='bold',
             transform=ax3.transAxes, zorder=4)

    fig.suptitle("Shor's Algorithm — Hasil Quantum  |  Qiskit 2.3.0",
                 fontsize=12, fontweight='bold', color='#e2e8f0', y=0.97)
    plt.savefig('shors_results.png', dpi=120, bbox_inches='tight', facecolor='#050d1a')
    plt.show()
    print("Disimpan ke shors_results.png ✅")

if p_factor and q_factor:
    plot_quantum_results(counts, n_count, N, a, r, p_factor, q_factor)


---
## 🔄 STEP 11 — Fungsi Shor Lengkap (End-to-End)

Membungkus seluruh alur Shor ke dalam satu fungsi utama dengan **retry loop** — karena algoritma Shor bersifat probabilistik dan mungkin perlu beberapa iterasi untuk menemukan period yang valid.

```
shors_algorithm(N, max_trials=10)
  └── Untuk setiap trial:
        ├── Pilih a acak
        ├── Cek lucky case (gcd ≠ 1)
        ├── Jalankan quantum circuit
        ├── Estimasi period r (CFE)
        ├── Validasi r
        └── Ekstraksi p, q via GCD
```

In [ ]:
# STEP 11: FUNGSI SHOR LENGKAP (END-TO-END)
def shors_algorithm(N, max_trials=10, shots=1024):
    if N % 2 == 0: return 2, N // 2
    if is_prime(N): return None, None

    n_c = 2 * ceil(log2(N))
    n_t = ceil(log2(N))
    sim = AerSimulator(method='statevector')

    for trial in range(1, max_trials + 1):
        a = random.randint(2, N - 1)
        g = gcd(a, N)
        if g != 1:
            print(f"Trial {trial}: lucky!  {N} = {g} × {N//g}")
            return g, N // g

        try:
            U  = build_mod_exp_oracle(a, N, n_c, n_t)
            qr_c = QuantumRegister(n_c, 'c')
            qr_t = QuantumRegister(n_t, 't')
            cr   = ClassicalRegister(n_c, 'm')
            qc   = QuantumCircuit(qr_c, qr_t, cr)
            qc.x(qr_t[0])
            for q in qr_c: qc.h(q)
            qc.append(UnitaryGate(U, label='Uₐ'), list(qr_c) + list(qr_t))
            qc.append(QFT(n_c, inverse=True, do_swaps=True), qr_c)
            qc.measure(qr_c, cr)
            cnt = dict(sorted(
                sim.run(transpile(qc, sim, optimization_level=1), shots=shots)
                   .result().get_counts().items(),
                key=lambda x: x[1], reverse=True
            ))
        except Exception as e:
            print(f"Trial {trial}: error — {e}"); continue

        r_found = None
        for bs in list(cnt.keys())[:10]:
            k = int(bs, 2)
            if k == 0: continue
            rc = Fraction(k / 2**n_c).limit_denominator(N).denominator
            for m in range(1, 8):
                if pow(a, rc * m, N) == 1:
                    r_found = rc * m; break
            if r_found: break

        if not r_found or r_found % 2 != 0: continue
        x = pow(a, r_found // 2, N)
        if x == N - 1: continue

        p, q = gcd(x - 1, N), gcd(x + 1, N)
        if p not in (1, N) and q not in (1, N):
            print(f"Trial {trial}: a={a}  r={r_found}  →  {N} = {p} × {q} ✅")
            return p, q
        print(f"Trial {trial}: a={a}  r={r_found}  faktor trivial, coba lagi...")

    return None, None

p_res, q_res = shors_algorithm(N=15, max_trials=10, shots=1024)


---
## 📋 STEP 12 — Ringkasan Akhir & Analisis Kompleksitas

### Kompleksitas Algoritma Shor

| Komponen | Klasik Terbaik (GNFS) | Shor's Algorithm |
|----------|----------------------|------------------|
| **Waktu** | $O\left(e^{(\log N)^{1/3}}\right)$ | $O\left((\log N)^3\right)$ |
| **Space** | $O\left(e^{(\log N)^{1/3}}\right)$ | $O\left((\log N)^2\right)$ qubit |
| **Gate** | — | $O(n^3)$ quantum gate |

### Implikasi Keamanan
> ⚠️ Algoritma Shor dapat memecahkan **RSA-2048** dalam waktu polinomial jika tersedia quantum computer dengan ~4,000 *logical qubit* (atau ~4 juta *physical qubit* dengan error correction).


In [ ]:
# STEP 12: ANALISIS KOMPLEKSITAS
import math

def complexity_analysis():
    N_list    = [2**k for k in range(4, 14)]
    log_N     = [math.log2(n) for n in N_list]
    shor_time = [(math.log2(n))**3 for n in N_list]
    gnfs_time = [math.exp((math.log(n)**(1/3)) * (math.log(math.log(n))**(2/3)))
                 for n in N_list]

    N_ex    = [15, 21, 35, 77, 143, 323, 1147]
    n_bits  = [ceil(log2(n)) for n in N_ex]
    n_cnt   = [2 * ceil(log2(n)) for n in N_ex]
    n_tgt   = [ceil(log2(n)) for n in N_ex]

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.patch.set_facecolor('#050d1a')
    for ax in axes:
        ax.set_facecolor('#0d1526'); ax.grid(alpha=0.2); ax.tick_params(colors='#94a3b8')

    # Plot kiri: kompleksitas waktu
    ax = axes[0]
    ax.semilogy(log_N, shor_time, 'o-',  color='#22d3ee', lw=2, ms=5, label='Shor O((log N)³)')
    ax.semilogy(log_N, gnfs_time, 's--', color='#ef4444', lw=2, ms=5, label='GNFS O(exp(...))', alpha=0.85)
    ax.set_xlabel('log₂(N)', fontsize=10); ax.set_ylabel('Kompleksitas (log scale)', fontsize=10)
    ax.set_title('Shor vs GNFS', fontsize=11, fontweight='bold', color='#e2e8f0')
    ax.legend(facecolor='#0d1526', edgecolor='#334155', labelcolor='#e2e8f0', fontsize=9)

    # Plot kanan: kebutuhan qubit
    ax = axes[1]
    x  = range(len(N_ex)); w = 0.35
    ax.bar([i - w/2 for i in x], n_cnt, w, color='#7c3aed', alpha=0.85, label='Counting')
    ax.bar([i + w/2 for i in x], n_tgt, w, color='#0369a1', alpha=0.85, label='Target')
    for i, (nc, nt) in enumerate(zip(n_cnt, n_tgt)):
        ax.text(i, nc + nt - nt + 0.3, f'{nc+nt}Q', ha='center',
                fontsize=8, color='#84cc16', fontweight='bold')
    ax.set_xticks(list(x))
    ax.set_xticklabels([f'N={n}\n({b}b)' for n, b in zip(N_ex, n_bits)], fontsize=8)
    ax.set_ylabel('Qubit', fontsize=10)
    ax.set_title('Kebutuhan Qubit per N', fontsize=11, fontweight='bold', color='#e2e8f0')
    ax.legend(facecolor='#0d1526', edgecolor='#334155', labelcolor='#e2e8f0', fontsize=9)

    fig.suptitle("Analisis Kompleksitas — Shor's Algorithm",
                 fontsize=12, fontweight='bold', color='#e2e8f0', y=1.01)
    plt.tight_layout()
    plt.savefig('shors_complexity.png', dpi=110, bbox_inches='tight', facecolor='#050d1a')
    plt.show()

complexity_analysis()

print(f"\nHasil akhir: {N} = {p_factor} × {q_factor}")
